# Chapter 9: Separation Technology and Equipment Design

This notebook demonstrates **three-phase separation** modeling using NeqSim,
covering separator performance at different operating pressures, the Souders-Brown
K-factor for separator sizing, multi-stage separation optimization, and flow
split visualization.

**Topics covered:**
- Gas-oil-water separation with ThreePhaseSeparator
- Phase split vs separator pressure
- K-factor (Souders-Brown) for separator sizing
- 3-stage separation optimization
- Flow distribution across separation stages

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    NEQSIM_MODE = "pip"

# Always ensure jneqsim is available (works in both modes)
try:
    import neqsim
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])

from neqsim import jneqsim
print(f"NeqSim ready (mode: {NEQSIM_MODE})")

# Common class shortcuts for convenience
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
SystemPrEos = jneqsim.thermo.system.SystemPrEos
SystemSrkCPAstatoil = jneqsim.thermo.system.SystemSrkCPAstatoil
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

# Process equipment
Stream = jneqsim.process.equipment.stream.Stream
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Compressor = jneqsim.process.equipment.compressor.Compressor
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Heater = jneqsim.process.equipment.heatexchanger.Heater
HeatExchanger = jneqsim.process.equipment.heatexchanger.HeatExchanger
Mixer = jneqsim.process.equipment.mixer.Mixer
Splitter = jneqsim.process.equipment.splitter.Splitter
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
Pump = jneqsim.process.equipment.pump.Pump
Expander = jneqsim.process.equipment.expander.Expander
Recycle = jneqsim.process.equipment.util.Recycle
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim2
Classpath:
  1. C:\Users\ESOL\Documents\GitHub\neqsim2\target\classes
  2. C:\Users\ESOL\Documents\GitHub\neqsim2\src\main\resources
  3. C:\Users\ESOL\Documents\GitHub\neqsim2\target\neqsim-3.7.0.jar



JVM started: C:\Users\ESOL\graalvm\graalvm-jdk-25.0.1+8.1\bin\server\jvm.dll
Ready — call neqsim_classes(ns) to import classes


All NeqSim classes imported OK
NeqSim loaded via devtools (local dev mode)
NeqSim ready (mode: devtools)


In [2]:
import matplotlib.pyplot as plt
import numpy as np

from neqsim import jneqsim

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
Separator = jneqsim.process.equipment.separator.Separator
ThrottlingValve = jneqsim.process.equipment.valve.ThrottlingValve
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

## 9.1 Create a Three-Phase (Gas-Oil-Water) Fluid

We model a typical production well stream containing gas, oil (represented
by nC10), and water.

In [3]:
def create_wellstream(T_C=60.0, P_bara=80.0):
    """Create a 3-phase wellstream fluid."""
    fluid = SystemSrkEos(273.15 + T_C, P_bara)
    fluid.addComponent("nitrogen", 0.01)
    fluid.addComponent("CO2", 0.02)
    fluid.addComponent("methane", 0.50)
    fluid.addComponent("ethane", 0.06)
    fluid.addComponent("propane", 0.04)
    fluid.addComponent("i-butane", 0.02)
    fluid.addComponent("n-butane", 0.03)
    fluid.addComponent("n-pentane", 0.02)
    fluid.addComponent("n-hexane", 0.02)
    fluid.addComponent("nC10", 0.18)
    fluid.addComponent("water", 0.10)
    fluid.setMixingRule("classic")
    fluid.setMultiPhaseCheck(True)
    return fluid

# Quick test
test_fluid = create_wellstream()
ops = ThermodynamicOperations(test_fluid)
ops.TPflash()
test_fluid.initProperties()
print(f"Number of phases: {test_fluid.getNumberOfPhases()}")
for i in range(test_fluid.getNumberOfPhases()):
    phase = test_fluid.getPhase(i)
    print(f"  Phase {i}: {phase.getPhaseTypeName()}, "
          f"density = {phase.getDensity('kg/m3'):.1f} kg/m3, "
          f"mole frac = {test_fluid.getMoleFraction(i):.4f}")

Number of phases: 3
  Phase 0: gas, density = 66.7 kg/m3, mole frac = 0.4779
  Phase 1: oil, density = 618.3 kg/m3, mole frac = 0.4302
  Phase 2: aqueous, density = 969.6 kg/m3, mole frac = 0.0919


## 9.2 Figure 1: Gas-Liquid Split vs Separator Pressure

We flash the wellstream at different pressures to see how the gas/oil/water
distribution changes. Lower pressures release more gas from the oil phase.

In [4]:
sep_pressures = np.array([10, 15, 20, 30, 40, 50, 60, 70, 80])
gas_fracs = []
oil_fracs = []
water_fracs = []

for P in sep_pressures:
    fluid = create_wellstream(60.0, float(P))
    feed = Stream("feed", fluid)
    feed.setFlowRate(50000.0, "kg/hr")
    feed.setTemperature(273.15 + 60.0, "K")
    feed.setPressure(float(P), "bara")

    sep = ThreePhaseSeparator("separator", feed)

    process = ProcessSystem()
    process.add(feed)
    process.add(sep)
    process.run()

    gas_rate = float(sep.getGasOutStream().getFlowRate("kg/hr"))
    oil_rate = float(sep.getOilOutStream().getFlowRate("kg/hr"))
    water_rate = float(sep.getWaterOutStream().getFlowRate("kg/hr"))
    total = gas_rate + oil_rate + water_rate

    gas_fracs.append(gas_rate / total * 100)
    oil_fracs.append(oil_rate / total * 100)
    water_fracs.append(water_rate / total * 100)

# Plot stacked area chart
fig, ax = plt.subplots(figsize=(9, 5))
ax.stackplot(sep_pressures, gas_fracs, oil_fracs, water_fracs,
             labels=['Gas', 'Oil', 'Water'],
             colors=['#2196F3', '#FF9800', '#4CAF50'], alpha=0.8)
ax.set_xlabel('Separator Pressure (bara)', fontsize=12)
ax.set_ylabel('Mass Fraction (%)', fontsize=12)
ax.set_title('Phase Split vs Separator Pressure\n(T = 60 °C, Wellstream)', fontsize=13)
ax.legend(loc='center right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(sep_pressures[0], sep_pressures[-1])
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_19764\2731538251.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** As separator pressure decreases, more gas evolves from the
oil phase (solution gas liberation), increasing the gas fraction and reducing
the oil fraction. The water fraction remains relatively constant since water
is essentially immiscible. Operating at very low pressure maximizes gas recovery
but can cause excessive oil shrinkage and volatile component loss.

## 9.3 Figure 2: Souders-Brown K-Factor vs Pressure

The Souders-Brown factor $K$ determines the maximum allowable gas velocity
in a separator:

$$V_{max} = K \sqrt{\frac{\rho_l - \rho_g}{\rho_g}}$$

We calculate gas and liquid densities at different separator pressures to
show how the velocity factor changes.

In [5]:
K_factor = 0.107  # Typical for horizontal separator with wire mesh demister (m/s)
k_pressures = np.array([10, 20, 30, 40, 50, 60, 70, 80])
gas_densities = []
oil_densities = []
max_velocities = []

for P in k_pressures:
    fluid = create_wellstream(60.0, float(P))
    ops = ThermodynamicOperations(fluid)
    ops.TPflash()
    fluid.initProperties()

    rho_g = float('nan')
    rho_l = float('nan')

    if fluid.hasPhaseType("gas"):
        rho_g = float(fluid.getPhase("gas").getDensity("kg/m3"))
    if fluid.hasPhaseType("oil"):
        rho_l = float(fluid.getPhase("oil").getDensity("kg/m3"))

    gas_densities.append(rho_g)
    oil_densities.append(rho_l)

    if not np.isnan(rho_g) and not np.isnan(rho_l) and rho_g > 0:
        v_max = K_factor * np.sqrt((rho_l - rho_g) / rho_g)
        max_velocities.append(v_max)
    else:
        max_velocities.append(float('nan'))

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: Densities
ax1.plot(k_pressures, gas_densities, 'b^-', linewidth=2, markersize=8, label='Gas density')
ax1.plot(k_pressures, oil_densities, 'ro-', linewidth=2, markersize=8, label='Oil density')
ax1.set_xlabel('Pressure (bara)', fontsize=12)
ax1.set_ylabel('Density (kg/m³)', fontsize=12)
ax1.set_title('Phase Densities vs Pressure\n(T = 60 °C)', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Right: Max velocity
ax2.plot(k_pressures, max_velocities, 'gs-', linewidth=2, markersize=8)
ax2.set_xlabel('Pressure (bara)', fontsize=12)
ax2.set_ylabel('Maximum Gas Velocity (m/s)', fontsize=12)
ax2.set_title(f'Souders-Brown Max Velocity vs Pressure\n(K = {K_factor} m/s)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_19764\3608377927.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** At higher pressures, gas density increases significantly while
oil density changes only slightly, reducing the density ratio and hence the
allowable gas velocity. This means high-pressure separators need larger
diameters to handle the same gas volume. Conversely, low-pressure separators
can tolerate higher velocities but must handle larger actual gas volumes.

## 9.4 Figure 3: Three-Stage Separation Optimization

Multi-stage separation recovers more oil (stock-tank liquid) than single-stage
flash by allowing gas to evolve gradually. We fix the 1st stage at 70 bara
and 3rd stage at 1.5 bara, then optimize the 2nd stage pressure.

In [6]:
P1 = 70.0   # 1st stage pressure (fixed)
P3 = 1.5    # 3rd stage / stock tank pressure (fixed)
second_stage_pressures = np.arange(5, 45, 3)  # 2nd stage pressure sweep
oil_yields = []

for P2 in second_stage_pressures:
    # Stage 1
    fluid = create_wellstream(60.0, P1)
    feed1 = Stream("feed1", fluid)
    feed1.setFlowRate(50000.0, "kg/hr")
    feed1.setTemperature(273.15 + 60.0, "K")
    feed1.setPressure(P1, "bara")

    sep1 = ThreePhaseSeparator("HP separator", feed1)

    # Valve to 2nd stage
    valve1 = ThrottlingValve("HP valve", sep1.getOilOutStream())
    valve1.setOutletPressure(float(P2))

    # Stage 2
    sep2 = ThreePhaseSeparator("MP separator", valve1.getOutletStream())

    # Valve to 3rd stage
    valve2 = ThrottlingValve("MP valve", sep2.getOilOutStream())
    valve2.setOutletPressure(P3)

    # Stage 3 (stock tank)
    sep3 = ThreePhaseSeparator("LP separator", valve2.getOutletStream())

    process = ProcessSystem()
    process.add(feed1)
    process.add(sep1)
    process.add(valve1)
    process.add(sep2)
    process.add(valve2)
    process.add(sep3)

    try:
        process.run()
        stock_tank_oil = float(sep3.getOilOutStream().getFlowRate("kg/hr"))
        oil_yields.append(stock_tank_oil)
    except Exception as e:
        print(f"  P2={P2:.0f} failed: {e}")
        oil_yields.append(float('nan'))

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(second_stage_pressures, [y/1000 for y in oil_yields], 'ko-',
        linewidth=2, markersize=8)

# Mark optimum
valid_yields = [(p, y) for p, y in zip(second_stage_pressures, oil_yields) if not np.isnan(y)]
if valid_yields:
    opt_p, opt_y = max(valid_yields, key=lambda x: x[1])
    ax.axvline(x=opt_p, color='red', linestyle='--', alpha=0.7)
    ax.annotate(f'Optimum: {opt_p:.0f} bara', xy=(opt_p, opt_y/1000),
                xytext=(opt_p + 5, opt_y/1000 + 0.1), fontsize=10, color='red',
                arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('2nd Stage Separator Pressure (bara)', fontsize=12)
ax.set_ylabel('Stock-Tank Oil Rate (tonnes/hr)', fontsize=12)
ax.set_title(f'3-Stage Separation Optimization\n'
             f'(P₁ = {P1:.0f} bara, P₃ = {P3:.1f} bara)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

C:\Users\ESOL\AppData\Local\Temp\ipykernel_19764\524396975.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** The optimal 2nd-stage pressure produces the maximum stock-tank
oil recovery. A common rule of thumb is equal pressure ratios between stages:
$P_2 = \sqrt{P_1 \cdot P_3}$. Too low a 2nd-stage pressure causes excessive
flash gas (liquid shrinkage), while too high a pressure doesn't liberate enough
gas before the final flash. The optimum typically corresponds to 1-2% more oil
recovery compared to the worst-case selection — significant at field scale.

## 9.5 Figure 4: Flow Distribution Across Separation Stages

We visualize how the total wellstream splits across the three separation
stages, using the optimal 2nd-stage pressure from above.

In [7]:
# Run the 3-stage separation at the optimal P2
if valid_yields:
    P2_opt = opt_p
else:
    P2_opt = 15.0

fluid = create_wellstream(60.0, P1)
feed1 = Stream("feed1", fluid)
feed1.setFlowRate(50000.0, "kg/hr")
feed1.setTemperature(273.15 + 60.0, "K")
feed1.setPressure(P1, "bara")

sep1 = ThreePhaseSeparator("HP separator", feed1)
valve1 = ThrottlingValve("HP valve", sep1.getOilOutStream())
valve1.setOutletPressure(float(P2_opt))
sep2 = ThreePhaseSeparator("MP separator", valve1.getOutletStream())
valve2 = ThrottlingValve("MP valve", sep2.getOilOutStream())
valve2.setOutletPressure(P3)
sep3 = ThreePhaseSeparator("LP separator", valve2.getOutletStream())

process = ProcessSystem()
process.add(feed1)
process.add(sep1)
process.add(valve1)
process.add(sep2)
process.add(valve2)
process.add(sep3)
process.run()

# Collect flow rates from each stage
stages = ['HP Sep\n(Stage 1)', 'MP Sep\n(Stage 2)', 'LP Sep\n(Stage 3)']
seps = [sep1, sep2, sep3]

gas_rates = []
oil_rates = []
water_rates = []

for sep in seps:
    gas_rates.append(float(sep.getGasOutStream().getFlowRate("kg/hr")) / 1000)
    oil_rates.append(float(sep.getOilOutStream().getFlowRate("kg/hr")) / 1000)
    water_rates.append(float(sep.getWaterOutStream().getFlowRate("kg/hr")) / 1000)

x = np.arange(len(stages))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
bars_gas = ax.bar(x - width, gas_rates, width, label='Gas', color='#2196F3', edgecolor='navy')
bars_oil = ax.bar(x, oil_rates, width, label='Oil', color='#FF9800', edgecolor='brown')
bars_water = ax.bar(x + width, water_rates, width, label='Water', color='#4CAF50', edgecolor='darkgreen')

ax.set_xlabel('Separation Stage', fontsize=12)
ax.set_ylabel('Flow Rate (tonnes/hr)', fontsize=12)
ax.set_title(f'Flow Split at Each Separation Stage\n'
             f'(P₁={P1:.0f}, P₂={P2_opt:.0f}, P₃={P3:.1f} bara)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(stages, fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars_gas, bars_oil, bars_water]:
    for bar in bars:
        height = bar.get_height()
        if height > 0.1:
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                    f'{height:.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

# Print summary table
print("\n=== Separation Train Summary ===")
print(f"{'Stage':<12} {'Pressure':>10} {'Gas (t/hr)':>12} {'Oil (t/hr)':>12} {'Water (t/hr)':>12}")
print("-" * 60)
pressures = [P1, P2_opt, P3]
for stage, P, g, o, w in zip(stages, pressures, gas_rates, oil_rates, water_rates):
    stage_clean = stage.replace('\n', ' ')
    print(f"{stage_clean:<12} {P:>8.1f} ba {g:>12.1f} {o:>12.1f} {w:>12.1f}")


=== Separation Train Summary ===
Stage          Pressure   Gas (t/hr)   Oil (t/hr) Water (t/hr)
------------------------------------------------------------
HP Sep (Stage 1)     70.0 ba         11.1         37.2          1.8
MP Sep (Stage 2)      8.0 ba          3.8         33.3          0.0
LP Sep (Stage 3)      1.5 ba          1.3         32.0          0.0


C:\Users\ESOL\AppData\Local\Temp\ipykernel_19764\2704065618.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Discussion:** The HP separator handles the bulk of gas separation at high
pressure, while successive stages liberate progressively less gas. Oil flow
decreases at each stage as dissolved gas is released. The water phase is
primarily removed at the first stage. This staged approach maximizes overall
oil recovery compared to a single-stage flash to atmospheric pressure.

## Summary

Key takeaways from this chapter:

1. **Separator pressure controls gas-oil split** — lower pressure liberates more
   gas but shrinks the oil.
2. **Souders-Brown K-factor** decreases with pressure due to increasing gas
   density, requiring larger separator vessels at high pressure.
3. **Multi-stage separation** optimizes oil recovery — the intermediate stage
   pressure has an optimum near $P_2 = \sqrt{P_1 \cdot P_3}$.
4. **Most gas is separated at the first (HP) stage**, with diminishing amounts
   at subsequent stages.